In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# 1. Setup Professional Logging (Industry Standard for tracking code execution)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def generate_rto_data(n_samples: int = 10000, output_dir: str = "../data", filename: str = "raw_rto_data.csv", seed: int = 42) -> pd.DataFrame:
    """
    Generates synthetic but highly realistic E-commerce RTO (Return to Origin) data.
    Uses fast array math (vectorized operations) instead of slow loops.
    """
    logger.info(f"⚙️ Initializing Data Generation Pipeline for {n_samples} samples...")
    
    # Set seed so we get the exact same random data every time we run this
    np.random.seed(seed)

    # 1. Create Order IDs (e.g., ORD_00001, ORD_00002)
    order_ids = [f"ORD_{str(i).zfill(5)}" for i in range(1, n_samples + 1)]

    # 2. Generate Cart Values (Most orders are around 1000-3000, some are very high)
    cart_values = np.random.lognormal(mean=6.5, sigma=0.8, size=n_samples).astype(int)
    cart_values = np.clip(cart_values, 250, 25000) # Keep values between ₹250 and ₹25,000

    # 3. Generate Past Return Rates for customers (Mostly low, some high)
    return_rates = np.round(np.random.beta(a=1, b=6, size=n_samples), 2)

    # 4. Generate Location and Payment Data
    pincode_tiers = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2], size=n_samples)
    address_quality = np.random.randint(1, 11, size=n_samples) # Score from 1 to 10
    is_cod = np.random.choice([0, 1], p=[0.35, 0.65], size=n_samples) # 65% are COD orders

    # 5. Core Business Logic (Calculate the actual risk of return for each order)
    base_prob = np.full(n_samples, 0.05) # Base 5% risk for everyone
    
    # Add risk if payment is Cash on Delivery (COD)
    base_prob += (is_cod == 1) * 0.15
    
    # Add risk based on their past return history
    base_prob += return_rates * 0.45
    
    # Increase risk for bad addresses in Tier 3 cities
    base_prob += np.where((pincode_tiers == 3) & (address_quality <= 4), 0.25, 0.0)
    
    # Decrease risk for great addresses in Tier 1 cities
    base_prob -= np.where((pincode_tiers == 1) & (address_quality >= 8), 0.05, 0.0)
    
    # Increase risk if the cart value is unusually high (Impulse buying)
    base_prob += np.where(cart_values > 10000, 0.10, 0.0)
        
    # Make sure probabilities stay strictly between 0% and 95%
    true_rto_prob = np.clip(base_prob, 0.0, 0.95)

    # 6. Final Target Variable: Will they actually return it? (1 = Yes, 0 = No)
    will_rto = np.random.binomial(n=1, p=true_rto_prob, size=n_samples)

    # 7. Combine everything into a Pandas DataFrame
    df = pd.DataFrame({
        'Order_ID': order_ids,
        'Cart_Value': cart_values,
        'Return_Rate': return_rates,
        'Pincode_Tier': pincode_tiers,
        'Address_Quality': address_quality,
        'Is_COD': is_cod,
        'Is_RTO': will_rto 
    })

    # 8. Save the file securely (Create the 'data' folder if it doesn't exist)
    save_path = Path(output_dir)
    save_path.mkdir(parents=True, exist_ok=True) 
    full_file_path = save_path / filename
    
    df.to_csv(full_file_path, index=False)
    
    logger.info(f"✅ Data saved securely at: {full_file_path}\n")
    return df

# Execution Block (This runs when you start the script)
if __name__ == "__main__":
    logger.info("🚀 Starting Data Generation Process...")
    
    # Step 1: Generate 10,000 rows to teach the Machine Learning model
    train_df = generate_rto_data(n_samples=10000, filename="raw_rto_data.csv", seed=42)
    
    # Step 2: Generate 100 rows for the UI Dashboard Batch Test
    test_df = generate_rto_data(n_samples=100, filename="test_batch_100_orders.csv", seed=99)
    
    logger.info("🎉 All data files are ready in the '../data' folder!")
    
    # Show the first 5 rows to verify
    try:
        display(train_df.head()) 
    except NameError:
        print(train_df.head())

2026-08-16 19:07:49,512 - INFO - 🚀 Starting Data Generation Process...
2026-08-16 19:07:49,513 - INFO - ⚙️ Initializing Data Generation Pipeline for 10000 samples...
2026-08-16 19:07:49,542 - INFO - ✅ Data saved securely at: ..\data\raw_rto_data.csv

2026-08-16 19:07:49,543 - INFO - ⚙️ Initializing Data Generation Pipeline for 100 samples...
2026-08-16 19:07:49,547 - INFO - ✅ Data saved securely at: ..\data\test_batch_100_orders.csv

2026-08-16 19:07:49,548 - INFO - 🎉 All data files are ready in the '../data' folder!


,Order_ID,Cart_Value,Return_Rate,Pincode_Tier,Address_Quality,Is_COD,Is_RTO
0,ORD_00001,989,0.06,3,1,1,0
1,ORD_00002,595,0.10,2,2,1,0
2,ORD_00003,1116,0.04,2,7,1,0
3,ORD_00004,2249,0.14,2,3,1,1
4,ORD_00005,551,0.08,1,7,0,0
